# 01 · Q – Question

Problem, Zielgruppe, Forschungsfragen, KPIs und geplante Bereitstellung.

**Projekt:** MietCheck · Data Analytics & Big Data

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

COLORS = {"navy": "#14213D", "blue": "#2563EB", "teal": "#0F766E",
          "amber": "#F59E0B", "red": "#DC2626", "grey": "#64748B"}
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.titleweight": "bold",
                     "axes.labelsize": 10, "figure.dpi": 110})

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

print(f"Projektwurzel: {ROOT}")

Projektwurzel: C:\Users\nelek\Desktop\Big Data\MietCheck


## Position im QUA³CK-Prozess

Jedes belastbare Data-Science-Projekt beginnt nicht mit einem Algorithmus, sondern mit einer klaren Entscheidung darüber, **welches Problem gelöst werden soll**. Die Q-Phase übersetzt den gesellschaftlichen Kontext steigender Wohnkosten in eine prüfbare Data-Science-Aufgabe. Sie definiert Zielgruppen, Forschungsfragen, Erfolgsmetriken, Grenzen und das Deployment-Ziel, bevor Daten oder Modelle ausgewählt werden.

> **Zentrale Forschungsfrage:** Wie groß ist die Lücke zwischen kleinräumiger Bestandsmiete und aktueller Angebotsmiete in Deutschland, welche räumlichen und wohnungsbezogenen Faktoren erklären die Bestandsmiete und wie zuverlässig lässt sie sich für eine konkrete Wohnsituation schätzen?

## Problem und Relevanz

Bestehende Mietverhältnisse, heutige Wohnungsangebote und die persönliche Vertragsmiete messen drei unterschiedliche Realitäten. Viele Rechner verdichten sie trotzdem zu einer einzigen vermeintlich „fairen“ Miete. Dadurch bleiben Zeitbezug, Marktstreuung und Modellfehler unsichtbar.

MietCheck trennt deshalb:

1. **Bestandsrealität:** amtliche Zensus-Nettokaltmiete zum 15.05.2022,
2. **Umzugsrealität:** aktuelle GREIX-Angebotsmiete bis Q1/2026,
3. **persönliche Realität:** eigene Vertragsmiete und Mietbelastung.

Der daraus abgeleitete **Umzugsaufschlag** beantwortet eine konkrete Alltagsfrage: Was würde ein heutiger Umzug gegenüber dem lokalen Wohnungsbestand und meiner jetzigen Situation finanziell verändern?

## Zielgruppen und Entscheidungssituationen

| Zielgruppe | Konkrete Entscheidung | Benötigte Information |
|---|---|---|
| Mieterinnen und Mieter | eigene Miete einordnen | persönliche Miete vs. lokaler Bestand |
| Wohnungssuchende | Umzug finanziell bewerten | aktueller Angebotsmedian und Markt-IQR |
| Studierende / Berufseinsteiger | Budgetfolgen abschätzen | monatliche Belastung relativ zum Einkommen |
| dateninteressierte Öffentlichkeit | regionale Unterschiede verstehen | Zeitreihen, Methodik, Quellen und Grenzen |

Die App ist bewusst **keine Rechtsprüfung**, kein amtlicher Mietspiegel und keine punktgenaue Wohnungsbewertung.

In [2]:
stakeholders = pd.DataFrame([
    ("Mietende", "aktuelle Vertragsmiete einordnen", "Bestandsanker + eigenes Mietbild"),
    ("Wohnungssuchende", "Umzugsfolgen abschätzen", "Angebotsmedian + Umzugsaufschlag"),
    ("Studierende / Berufseinsteiger", "Budgetrisiko prüfen", "Mietbelastung am Haushaltsnetto"),
    ("Öffentlichkeit", "regionale Unterschiede verstehen", "Zeitreihe + Methodik + Grenzen"),
], columns=["Zielgruppe", "Entscheidung", "App-Ausgabe"])
display(stakeholders.style.hide(axis="index"))

Zielgruppe,Entscheidung,App-Ausgabe
Mietende,aktuelle Vertragsmiete einordnen,Bestandsanker + eigenes Mietbild
Wohnungssuchende,Umzugsfolgen abschätzen,Angebotsmedian + Umzugsaufschlag
Studierende / Berufseinsteiger,Budgetrisiko prüfen,Mietbelastung am Haushaltsnetto
Öffentlichkeit,regionale Unterschiede verstehen,Zeitreihe + Methodik + Grenzen


## Thematische Forschungsfragen

1. Wie stark variiert die Bestandsmiete innerhalb und zwischen deutschen Regionen?
2. Verbessern strukturelle Wohnumfeldmerkmale eine rein kategoriale Baseline?
3. Welches Regressionsverfahren generalisiert auf bislang ungesehene räumliche Gebiete?
4. Wie groß ist die Differenz zwischen Zensus-Bestandsmiete 2022 und GREIX-Angebotsmiete 2026?
5. Wie verändert diese Differenz die persönliche Mietbelastung bei einem Umzug?
6. Wie muss Unsicherheit dargestellt werden, damit aus einem Modellwert keine Scheingenauigkeit wird?

## Vorab definierte KPIs und Go/No-Go-Regeln

| Dimension | Ziel vor Modellierung | Konsequenz bei Nichterfüllung |
|---|---|---|
| Big Data | mindestens 2 Mio. reproduzierbare Modellzeilen | Datenkonzept neu bewerten |
| Modellvergleich | mindestens vier Modellfamilien plus fachliche Baseline | A¹ erweitern |
| Generalisierung | räumlich disjunkte Entwicklung, Kalibrierung und Test | kein Deployment |
| Modellnutzen | mindestens 15 % niedrigerer Test-MAE als Baseline | Produkt-Hook neu bewerten |
| Unsicherheit | separates Kalibrierungsset und gemessene Coverage | keine Punktzahl ohne Warnung |
| Produkt | zentrale Nutzerreise unter zwei Sekunden und mobil lesbar | Optimierung vor Freigabe |
| Reproduzierbarkeit | Download, Tests, GitHub Actions und ausgeführte Notebooks | keine Abgabe |

Die Schwellenwerte werden **vor** dem finalen Test festgelegt. Dadurch kann ein Ergebnis nicht nachträglich passend erklärt werden.

In [3]:
criteria = pd.DataFrame([
    ("Datenumfang", "≥ 2.000.000 Zeilen", "reports/dataset_build_report.json"),
    ("Modellvergleich", "≥ 4 Familien + Baseline", "reports/algorithm_benchmark.json"),
    ("Testdesign", "0 gemeinsame 25-km-Blöcke", "reports/hgb_tuning.json"),
    ("Modellnutzen", "≥ 15 % MAE-Verbesserung", "reports/final_model_evaluation.json"),
    ("Unsicherheit", "separate Kalibrierung + Coverage", "reports/final_model_evaluation.json"),
    ("Deployment", "Streamlit + GitHub + CI", "app.py / .github/workflows/ci.yml"),
], columns=["Dimension", "Erfolgskriterium", "späterer Nachweis"])
display(criteria.style.hide(axis="index"))

Dimension,Erfolgskriterium,späterer Nachweis
Datenumfang,≥ 2.000.000 Zeilen,reports/dataset_build_report.json
Modellvergleich,≥ 4 Familien + Baseline,reports/algorithm_benchmark.json
Testdesign,0 gemeinsame 25-km-Blöcke,reports/hgb_tuning.json
Modellnutzen,≥ 15 % MAE-Verbesserung,reports/final_model_evaluation.json
Unsicherheit,separate Kalibrierung + Coverage,reports/final_model_evaluation.json
Deployment,Streamlit + GitHub + CI,app.py / .github/workflows/ci.yml


## Marktumfeld und Alleinstellungsmerkmal

Amtliche Mietspiegel, Immobilienportale und Budgetrechner beantworten jeweils Teilfragen. MietCheck verbindet in **einer quelloffenen Nutzerreise** einen kleinräumigen ML-Bestandsanker, einen aktuellen unabhängigen Angebotsmarkt und die persönliche Mietbelastung. Entscheidend ist nicht ein weiterer Preisrechner, sondern die methodische Trennung:

- **Zensus:** Was wurde im Bestand 2022 durchschnittlich gezahlt?
- **GREIX:** Was wird bei heutigen Angeboten in einem Markt verlangt?
- **persönliche Eingabe:** Was bedeutet der Abstand für diesen Haushalt?

Diese Kombination ist der USP; jede Einzelkomponente bleibt hinsichtlich Datenstand, Abdeckung und Unsicherheit sichtbar.

## Deployment-Ziel und Lieferobjekte

Das Ergebnis der K-Phase ist eine responsive Streamlit-App mit reservierter URL `mietcheck.streamlit.app`, ein öffentlich nachvollziehbares GitHub-Repository, vollständig ausgeführte QUA³CK-Notebooks, Modell- und Datenkarte, automatisierte Tests sowie Präsentations- und Dokumentationsartefakte.

**Bewusste Grenzen:** keine Portal-Scrapes, keine personenbezogene Speicherung, keine Rechts- oder Finanzberatung, keine lokale Angebotsbehauptung außerhalb der 37 belegten GREIX-Märkte.

In [4]:
deliverables = pd.DataFrame([
    ("ML-Nachweis", "ausgeführte QUA³CK-Notebooks + versionierte Reports"),
    ("Produkt", "responsive Streamlit-App"),
    ("Reproduzierbarkeit", "GitHub, Downloadskripte, Tests und CI"),
    ("Transparenz", "Datenkarte, Modellkarte, Risiko-/Ethikdokument"),
    ("Prüfung", "Präsentation, schriftliche Ausarbeitung und Demo-Runbook"),
], columns=["Lieferobjekt", "Abnahmekriterium"])
display(deliverables.style.hide(axis="index"))

Lieferobjekt,Abnahmekriterium
ML-Nachweis,ausgeführte QUA³CK-Notebooks + versionierte Reports
Produkt,responsive Streamlit-App
Reproduzierbarkeit,"GitHub, Downloadskripte, Tests und CI"
Transparenz,"Datenkarte, Modellkarte, Risiko-/Ethikdokument"
Prüfung,"Präsentation, schriftliche Ausarbeitung und Demo-Runbook"


---

**Reproduzierbarkeit:** Die visualisierten Kennzahlen stammen aus versionierten JSON-/CSV-Artefakten. Die jeweils genannten Skripte erzeugen diese Artefakte aus den öffentlichen Rohdaten erneut. Relative Pfade funktionieren sowohl aus der Projektwurzel als auch aus `notebooks/`.